# Download mmCIF files and build protein contact graphs

This notebook reads the chain-level manifest from notebook 01, downloads one compressed mmCIF file per PDB entry from RCSB, builds one C-alpha contact graph per selected chain, and stores each NetworkX graph as an atomic pickle cache. The status TSV and JSONL event log let interrupted runs resume.

A tqdm progress bar advances after each protein is created, reused from cache, or recorded as failed.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

GRAPHICALIZER_SRC = PROJECT_ROOT.parent / "abstractgraph-ecosystem/repos/abstractgraph-graphicalizer/src"
if GRAPHICALIZER_SRC.exists():
    sys.path.insert(0, str(GRAPHICALIZER_SRC))

from enzyme_classification.graphs import build_protein_graph_dataset

DATA_DIR = PROJECT_ROOT / "data"
EC_LEVEL = 1  # Keep this aligned with notebook 01.
MANIFEST_PATH = DATA_DIR / "interim" / f"ec_protein_manifest_level_{EC_LEVEL}.tsv"
RAW_MMCIF_DIR = DATA_DIR / "raw" / "mmcif"
GRAPH_DIR = DATA_DIR / "processed" / "graphs"
STATUS_PATH = DATA_DIR / "processed" / f"graph_build_status_level_{EC_LEVEL}.tsv"
EVENT_LOG_PATH = DATA_DIR / "processed" / f"graph_build_events_level_{EC_LEVEL}.jsonl"

MIN_CONTACT_DISTANCE = 4.0
MAX_CONTACT_DISTANCE = 8.0
OVERWRITE = False
SAMPLE_LIMIT = None  # Set to a small integer for a quick first run.
DOWNLOAD_TIMEOUT_SECONDS = 60
DOWNLOAD_MAX_RETRIES = 4
DOWNLOAD_RETRY_BACKOFF_SECONDS = 2.0

In [2]:
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Run notebook 01 first; missing {MANIFEST_PATH}")

manifest = pd.read_csv(MANIFEST_PATH, sep="\t", dtype=str).fillna("")
print(f"Manifest samples: {len(manifest):,}")
display(manifest.groupby("ec_number").size().rename("samples").to_frame())

def _cached_graph_path(sample_id):
    safe_id = str(sample_id).replace("/", "_").replace(chr(92), "_")
    return GRAPH_DIR / f"{safe_id}.pkl"

expected_graphs = [_cached_graph_path(row.sample_id) for row in manifest.itertuples()]
cached_graph_count = sum(path.exists() and path.stat().st_size > 0 for path in expected_graphs)
expected_mmcif = {str(pdb_id).lower() + ".cif.gz" for pdb_id in manifest["pdb_id"]}
cached_mmcif_count = sum((RAW_MMCIF_DIR / filename).exists() for filename in expected_mmcif)
print(f"Existing graph caches: {cached_graph_count:,}/{len(expected_graphs):,}")
print(f"Existing mmCIF files: {cached_mmcif_count:,}/{len(expected_mmcif):,} unique PDB entries")
print(f"OVERWRITE={OVERWRITE}; cached graphs and mmCIF files will be reused when possible")

Manifest samples: 35


,samples
ec_number,
1,5
2,5
3,5
4,5
5,5
6,5
7,5


In [ ]:
planned_samples = len(manifest) if SAMPLE_LIMIT is None else min(len(manifest), SAMPLE_LIMIT)
status_rows = []
with tqdm(total=planned_samples, desc="Building contact graphs", unit="protein") as progress_bar:
    def report_progress(result):
        progress_bar.update(1)
        progress_bar.set_postfix(
            status=result["status"],
            sample=result["sample_id"],
            refresh=True,
        )

    status_rows = build_protein_graph_dataset(
        MANIFEST_PATH,
        raw_mmcif_dir=RAW_MMCIF_DIR,
        graph_dir=GRAPH_DIR,
        status_path=STATUS_PATH,
        event_log_path=EVENT_LOG_PATH,
        min_distance=MIN_CONTACT_DISTANCE,
        max_distance=MAX_CONTACT_DISTANCE,
        overwrite=OVERWRITE,
        download_timeout_seconds=DOWNLOAD_TIMEOUT_SECONDS,
        download_max_retries=DOWNLOAD_MAX_RETRIES,
        download_retry_backoff_seconds=DOWNLOAD_RETRY_BACKOFF_SECONDS,
        sample_limit=SAMPLE_LIMIT,
        progress=report_progress,
        verbose=False,
    )
status = pd.DataFrame(status_rows)
print(status["status"].value_counts(dropna=False))
display(status.head())

created: 4b4o.H -> ./data/processed/graphs/4b4o.H.pkl
created: 3lc4.B -> ./data/processed/graphs/3lc4.B.pkl
created: 6sy1.B -> ./data/processed/graphs/6sy1.B.pkl
created: 7mjc.B -> ./data/processed/graphs/7mjc.B.pkl
created: 2dfd.D -> ./data/processed/graphs/2dfd.D.pkl
created: 3mdy.B -> ./data/processed/graphs/3mdy.B.pkl
created: 1ehw.B -> ./data/processed/graphs/1ehw.B.pkl
created: 4j9g.E -> ./data/processed/graphs/4j9g.E.pkl
created: 2a7l.B -> ./data/processed/graphs/2a7l.B.pkl
created: 6w2l.B -> ./data/processed/graphs/6w2l.B.pkl
created: 1mza.A -> ./data/processed/graphs/1mza.A.pkl
created: 6q3v.A -> ./data/processed/graphs/6q3v.A.pkl
created: 8yl3.A -> ./data/processed/graphs/8yl3.A.pkl
created: 4ikc.A -> ./data/processed/graphs/4ikc.A.pkl
created: 4iqy.B -> ./data/processed/graphs/4iqy.B.pkl
created: 3gvr.A -> ./data/processed/graphs/3gvr.A.pkl
created: 2bex.D -> ./data/processed/graphs/2bex.D.pkl
created: 3uvj.C -> ./data/processed/graphs/3uvj.C.pkl
created: 8trl.H -> ./data/pr

Each graph is an undirected NetworkX graph with one node per C-alpha residue and edges for contacts in the inclusive 4–8 Å window. Raw graphs retain measured edge distances; notebook 03 derives vectorizer-ready edge labels without changing this cache.